In [1]:
import json
import pandas as pd


In [2]:
with open(r"C:\Users\Mi\Downloads\export (2).geojson", "r", encoding="utf-8") as f:
    data = json.load(f)

In [3]:
rows = []

for feature in data["features"]:
    props = feature.get("properties", {})
    geom = feature.get("geometry", {})

    coords = geom.get("coordinates", [None, None])

    rows.append({
        "name": props.get("name"),
        "network": props.get("network"),
        "start_date": props.get("start_date"),
        "longitude": coords[0] if len(coords) > 1 else None,
        "latitude": coords[1] if len(coords) > 1 else None,
    })

df = pd.DataFrame(rows)

print(df.head())

             name                  network  start_date  longitude   latitude
0      Медведково  Московский метрополитен  1978-09-30  37.661550  55.887177
1    Бабушкинская  Московский метрополитен  1978-09-30  37.664110  55.869634
2    Партизанская  Московский метрополитен  1944-01-18  37.750993  55.788503
3     Семёновская  Московский метрополитен  1944-01-18  37.721282  55.783307
4  Филёвский парк  Московский метрополитен  1961-10-13  37.483372  55.739508


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 302 entries, 0 to 301
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        302 non-null    object 
 1   network     302 non-null    object 
 2   start_date  271 non-null    object 
 3   longitude   302 non-null    float64
 4   latitude    302 non-null    float64
dtypes: float64(2), object(3)
memory usage: 11.9+ KB


In [8]:
df2 = pd.read_csv(r"C:\Users\Mi\Downloads\zk_coords.csv")

In [9]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1035 entries, 0 to 1034
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      1035 non-null   int64  
 1   lat     1035 non-null   float64
 2   lng     1035 non-null   float64
dtypes: float64(2), int64(1)
memory usage: 24.4 KB


In [10]:
import pandas as pd
from sklearn.neighbors import BallTree
import numpy as np

# координаты станций
stations = np.deg2rad(
    df[['latitude', 'longitude']].values
)

# координаты объектов
objects = np.deg2rad(
    df2[['lat', 'lng']].values
)

# создаем дерево
tree = BallTree(stations, metric='haversine')

# ищем ближайшую станцию
distances, indexes = tree.query(objects, k=1)

# перевод расстояния в метры
df2['metro_distance_m'] = distances[:, 0] * 6371000

# добавляем название станции
df2['nearest_metro'] = df.iloc[indexes[:, 0]]['name'].values

# добавляем сеть (МЦК/метро)
df2['network'] = df.iloc[indexes[:, 0]]['network'].values

In [19]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1035 entries, 0 to 1034
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   id                       1035 non-null   int64   
 1   lat                      1035 non-null   float64 
 2   lng                      1035 non-null   float64 
 3   metro_distance_m         1035 non-null   float64 
 4   nearest_metro            1035 non-null   object  
 5   network                  1035 non-null   object  
 6   metro_distance_category  1035 non-null   category
dtypes: category(1), float64(3), int64(1), object(2)
memory usage: 49.9+ KB


In [16]:
df2['metro_distance_category'] = pd.cut(
    df2['metro_distance_m'],
    bins=[0, 500, 1000, 1500, 3000, 1000000],
    labels=[
        'до 500 м',
        '500-1000 м',
        '1-1.5 км',
        '1.5-3 км',
        'более 3 км'
    ]
)

In [18]:
df2.to_csv(r"C:\Users\Mi\Downloads\zk_coords_with_metro.csv", index=False)